# Evidence Detection (ED) — DeBERTa v3 5-seed ensemble
Binary classification for (claim, evidence) pairs using a 5-seed DeBERTa v3 ensemble with averaged probabilities and tuned threshold.


## Install and Imports


In [ ]:
# In a fresh environment install the required packages first:
# pip install pandas numpy scikit-learn torch transformers datasets accelerate sentencepiece protobuf tiktoken
import os, re, random, json
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)


def has_mps() -> bool:
    return (
        hasattr(torch.backends, "mps")
        and torch.backends.mps.is_built()
        and torch.backends.mps.is_available()
    )


DEVICE = "cuda" if torch.cuda.is_available() else "mps" if has_mps() else "cpu"
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score

# for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Using device:", DEVICE)


## Model Training


### Config and load data


In [ ]:
CONFIG = {
    "model_name": "microsoft/deberta-v3-base",
    "max_length": 256,
    "train_bs": 16,
    "eval_bs": 32,
    "infer_eval_bs": 64,
    "grad_accum": 1,
    "lr": 2e-5,
    "epochs": 3,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "seeds": [42, 52, 62, 72, 82],
    "out_dir": "./outputs/deberta_v3_5_seed_ensemble",
    "ensemble_info_path": "./outputs/deberta_v3_5_seed_ensemble/ensemble_info.json",
    "train_path": "./training_data/train.csv",
    "dev_path": "./training_data/dev.csv",
}

TEST_MODE = False  # set True for quick testing, False for full training

if DEVICE == "mps":
    CONFIG["train_bs"] = 2
    CONFIG["eval_bs"] = 4
    CONFIG["infer_eval_bs"] = 8
    CONFIG["grad_accum"] = 8
elif DEVICE == "cpu":
    CONFIG["train_bs"] = 4
    CONFIG["eval_bs"] = 8
    CONFIG["infer_eval_bs"] = 16
    CONFIG["grad_accum"] = 4

# load data into dataframes
train_df = pd.read_csv(CONFIG["train_path"])
dev_df   = pd.read_csv(CONFIG["dev_path"])

# ensure labels are integers
train_df["label"] = train_df["label"].astype(int)
dev_df["label"]   = dev_df["label"].astype(int)

if TEST_MODE:
    train_df = train_df.sample(128, random_state=SEED).reset_index(drop=True)
    dev_df   = dev_df.sample(128, random_state=SEED).reset_index(drop=True)

# print dataset stats
print("Train:", len(train_df), "Dev:", len(dev_df))
print("Train label dist:", train_df["label"].value_counts().to_dict())
print("Dev label dist:", dev_df["label"].value_counts().to_dict())
print("Runtime config:", {k: CONFIG[k] for k in ["train_bs", "eval_bs", "infer_eval_bs", "grad_accum", "max_length"]})
train_df.head(3)


### Tokenisation


In [ ]:
# create datasets and tokeniser
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], use_fast=False)
train_ds = Dataset.from_pandas(train_df[["Claim", "Evidence", "label"]], preserve_index=False)
dev_ds   = Dataset.from_pandas(dev_df[["Claim", "Evidence", "label"]], preserve_index=False)

# tokenize the datasets
def tok_fn(batch):
    return tokenizer(
        batch["Claim"],
        batch["Evidence"],
        truncation=True,
        max_length=CONFIG["max_length"],
    )

train_tok = train_ds.map(tok_fn, batched=True, remove_columns=["Claim", "Evidence"])
dev_tok   = dev_ds.map(tok_fn, batched=True, remove_columns=["Claim", "Evidence"])

# pads inputs to the same length in a batch
collator = DataCollatorWithPadding(tokenizer=tokenizer)
dev_labels = dev_df["label"].to_numpy()


### Training and metrics


In [ ]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def logits_to_pos_probs(logits: np.ndarray) -> np.ndarray:
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_vals = np.exp(shifted)
    probs = exp_vals / np.sum(exp_vals, axis=1, keepdims=True)
    return probs[:, 1]


def binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}


def tune_threshold(y_true: np.ndarray, probs: np.ndarray) -> tuple[float, float]:
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 181):
        preds = (probs >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = float(t), float(f1)
    return best_t, best_f1


def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    probs = logits_to_pos_probs(logits)
    preds = (probs >= 0.5).astype(int)
    return binary_metrics(y_true, preds)


seed_model_dirs = []
dev_prob_list = []

for seed in CONFIG["seeds"]:
    set_all_seeds(seed)

    run_dir = os.path.join(CONFIG["out_dir"], f"seed_{seed}")
    best_dir = os.path.join(run_dir, "best_model")
    os.makedirs(run_dir, exist_ok=True)

    # load pretrained DeBERTa v3 with a classification head for 2 labels
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["model_name"],
        num_labels=2,
        id2label={0: "not_relevant", 1: "relevant"},
        label2id={"not_relevant": 0, "relevant": 1},
    )

    args = TrainingArguments(
        output_dir=run_dir,
        learning_rate=CONFIG["lr"],
        per_device_train_batch_size=CONFIG["train_bs"],
        per_device_eval_batch_size=CONFIG["eval_bs"],
        gradient_accumulation_steps=CONFIG["grad_accum"],
        max_steps=2 if TEST_MODE else -1,
        num_train_epochs=CONFIG["epochs"] if not TEST_MODE else 1,
        weight_decay=CONFIG["weight_decay"],
        warmup_ratio=CONFIG["warmup_ratio"] if not TEST_MODE else 0.0,
        eval_strategy="epoch",
        save_strategy="epoch" if not TEST_MODE else "steps",
        save_steps=1 if TEST_MODE else None,
        load_best_model_at_end=True if not TEST_MODE else False,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=1,
        fp16=torch.cuda.is_available() if not TEST_MODE else False,
        dataloader_pin_memory=torch.cuda.is_available(),
        report_to="none",
        seed=seed,
        data_seed=seed,
        disable_tqdm=True,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # gather dev probabilities for ensemble averaging
    dev_pred = trainer.predict(dev_tok)
    dev_probs = logits_to_pos_probs(dev_pred.predictions)
    dev_prob_list.append(dev_probs)

    # save per-seed model + tokenizer
    trainer.save_model(best_dir)
    tokenizer.save_pretrained(best_dir)
    seed_model_dirs.append(best_dir)

    seed_preds = (dev_probs >= 0.5).astype(int)
    seed_stats = binary_metrics(dev_labels, seed_preds)
    print(f"Seed {seed} @0.5 metrics:", seed_stats)

ensemble_dev_probs = np.mean(np.vstack(dev_prob_list), axis=0)
best_threshold, best_dev_f1 = tune_threshold(dev_labels, ensemble_dev_probs)
dev_preds = (ensemble_dev_probs >= best_threshold).astype(int)
dev_stats = binary_metrics(dev_labels, dev_preds)

print(f"Tuned threshold (dev): {best_threshold:.3f}  |  F1={best_dev_f1:.4f}")
print("Ensemble dev metrics with tuned threshold:", dev_stats)


### Save model


In [ ]:
# Save ensemble metadata (all seed models are already saved in-run)
os.makedirs(CONFIG["out_dir"], exist_ok=True)
ensemble_info = {
    "model_name": CONFIG["model_name"],
    "strategy": "5_seed",
    "seeds": CONFIG["seeds"],
    "model_dirs": seed_model_dirs,
    "threshold": float(best_threshold),
    "max_length": CONFIG["max_length"],
}

with open(CONFIG["ensemble_info_path"], "w") as f:
    json.dump(ensemble_info, f, indent=2)

print("Saved ensemble metadata to:", CONFIG["ensemble_info_path"])
ensemble_info


## Model Demo


In [ ]:
# specify test file path and output path for predictions
TEST_PATH = "./test.csv"
OUT_PATH  = "./predictions_5_seed_ensemble.csv"


@torch.inference_mode()  # disables gradient tracking to make inference faster/cheaper
def predict_test_ensemble(model_dirs: list[str], threshold: float, test_csv: str, out_csv: str) -> pd.DataFrame:
    # load tokenizer from first seed model
    tok = AutoTokenizer.from_pretrained(model_dirs[0], use_fast=False)

    # load test file
    df = pd.read_csv(test_csv)

    # convert to HF dataset
    ds = Dataset.from_pandas(df[["Claim", "Evidence"]], preserve_index=False)

    # tokenise pairs in same way as training
    def tok_fn(batch):
        return tok(batch["Claim"], batch["Evidence"], truncation=True, max_length=CONFIG["max_length"])

    ds_tok = ds.map(tok_fn, batched=True, remove_columns=["Claim", "Evidence"])

    # pad batches
    data_collator = DataCollatorWithPadding(tokenizer=tok)

    # average probabilities from all seed models
    all_probs = []
    for model_dir in model_dirs:
        mdl = AutoModelForSequenceClassification.from_pretrained(model_dir)
        infer_args = TrainingArguments(
            output_dir=os.path.join(model_dir, "_infer_tmp"),
            per_device_eval_batch_size=CONFIG["infer_eval_bs"],
            dataloader_pin_memory=torch.cuda.is_available(),
            report_to="none",
            disable_tqdm=True,
        )
        infer_trainer = Trainer(model=mdl, args=infer_args, processing_class=tok, data_collator=data_collator)
        pred = infer_trainer.predict(ds_tok)
        probs = logits_to_pos_probs(pred.predictions)
        all_probs.append(probs)

    mean_probs = np.mean(np.vstack(all_probs), axis=0)
    yhat = (mean_probs >= threshold).astype(int)

    # write output CSV
    out = df.copy()
    out["prob_relevant"] = mean_probs
    out["pred"] = yhat
    out.to_csv(out_csv, index=False)

    return out


# Run demo inference and preview results
demo = predict_test_ensemble(seed_model_dirs, best_threshold, TEST_PATH, OUT_PATH)
demo.head(5), print("Wrote:", OUT_PATH)
